# Line Detection with a Calibrated Camera

**Goal:** use a calibrated laptop camera to detect straight lines more reliably.

This notebook is a follow-up to **Camera Calibration with ROS 2**.

By the end, you should understand:

1. Why lens distortion affects line detection.
2. How to load a ROS camera calibration YAML file.
3. How to undistort an image.
4. How to detect edges using Canny.
5. How to detect straight lines using the Hough Transform.
6. How to compare line detection before and after undistortion.

---

## Why this is useful in robotics

Robots often use line-like structures such as walls and lanes. If a camera has lens distortion, straight real-world lines can appear curved in the image. Calibration helps correct this geometry.


# 1. Background

In an ideal pinhole camera, a straight 3D line projects to a straight 2D line in the image. Real lenses introduce distortion, especially near image borders. This can make line detection less reliable. In this notebook we compare raw image line detection and undistorted image line detection to see why calibration improves geometric perception.


# 2. Vision pipeline

We will use this pipeline:

```text
Camera image
→ Undistortion
→ Grayscale conversion
→ Gaussian blur
→ Canny edge detection (finds pixels where intensity changes strongly. These are often object boundaries.)
→ Hough line detection (finds groups of edge pixels that form straight line segments.)
    OpenCV returns line segments as: (x1, y1, x2, y2)
→ Visualization
```




# 3. Import libraries

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import yaml
import matplotlib.pyplot as plt

print("OpenCV version:", cv2.__version__)

# 4. Helper functions

In [ ]:
def load_ros_camera_calibration(yaml_path):
    """Load camera calibration from a ROS-style camera YAML file."""
    yaml_path = Path(yaml_path).expanduser()
    if not yaml_path.exists():
        raise FileNotFoundError(f"Calibration file not found: {yaml_path}")

    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)

    camera_matrix = np.array(data["camera_matrix"]["data"], dtype=np.float64).reshape(3, 3)
    dist_coeffs = np.array(data["distortion_coefficients"]["data"], dtype=np.float64).reshape(-1, 1)
    image_width = data.get("image_width", None)
    image_height = data.get("image_height", None)

    return camera_matrix, dist_coeffs, image_width, image_height


def show_image_bgr(image_bgr, title="Image", figsize=(9, 6)):
    """Display an OpenCV BGR image in a notebook."""
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(image_rgb)
    plt.title(title)
    plt.axis("off")
    plt.show()


def show_gray_image(image_gray, title="Image", figsize=(9, 6)):
    """Display a grayscale image in a notebook."""
    plt.figure(figsize=figsize)
    plt.imshow(image_gray, cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.show()


def undistort_image(image_bgr, camera_matrix, dist_coeffs):
    """Undistort an image using camera calibration."""
    h, w = image_bgr.shape[:2]

    new_camera_matrix, roi = cv2.getOptimalNewCameraMatrix(
        camera_matrix,
        dist_coeffs,
        (w, h),
        alpha=1,
        newImgSize=(w, h)
    )

    undistorted = cv2.undistort(
        image_bgr,
        camera_matrix,
        dist_coeffs,
        None,
        new_camera_matrix
    )

    return undistorted, new_camera_matrix, roi


def preprocess_for_edges(image_bgr, blur_kernel_size=5):
    """Convert image to grayscale and apply Gaussian blur."""
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

    if blur_kernel_size > 0:
        gray = cv2.GaussianBlur(gray, (blur_kernel_size, blur_kernel_size), 0)

    return gray


def detect_edges(gray_image, low_threshold=50, high_threshold=150):
    """Run Canny edge detection."""
    return cv2.Canny(gray_image, low_threshold, high_threshold)


def detect_lines_hough(edges, threshold=80, min_line_length=60, max_line_gap=10):
    """Detect line segments using Probabilistic Hough Transform."""
    return cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi / 180,
        threshold=threshold,
        minLineLength=min_line_length,
        maxLineGap=max_line_gap
    )


def draw_lines(image_bgr, lines, color=(0, 255, 0), thickness=2):
    """Draw line segments on a copy of the image."""
    output = image_bgr.copy()

    if lines is None:
        return output

    for line in lines:
        x1, y1, x2, y2 = line[0]
        cv2.line(output, (x1, y1), (x2, y2), color, thickness)

    return output


def count_lines(lines):
    """Return the number of line segments."""
    return 0 if lines is None else len(lines)


def line_lengths(lines):
    """Compute lengths of detected line segments."""
    if lines is None:
        return []

    lengths = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        lengths.append(np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2))

    return lengths


def summarize_lines(lines, name="Lines"):
    """Print a summary of detected lines."""
    n = count_lines(lines)
    lengths = line_lengths(lines)

    print(f"{name}:")
    print(f"  Number of detected line segments: {n}")

    if n > 0:
        print(f"  Average line length: {np.mean(lengths):.1f} pixels")
        print(f"  Longest line length: {np.max(lengths):.1f} pixels")
    print()


# 5. Load your camera calibration

Use the calibration YAML file from Part A.

Expected path:

```text
~/camera_calib_ws/calibration/laptop_camera.yaml
```

Change the path below if your file is elsewhere.


In [ ]:
CALIBRATION_FILE = "~/camera_calib_ws/calibration/laptop_camera.yaml"

camera_matrix, dist_coeffs, image_width, image_height = load_ros_camera_calibration(CALIBRATION_FILE)

print("Camera matrix K:")
print(camera_matrix)

print("\nDistortion coefficients:")
print(dist_coeffs.ravel())

print("\nCalibration image size:")
print("width:", image_width)
print("height:", image_height)

# 6. Capture an image from the webcam

Point your camera at a scene with many straight lines.

Good examples:

- door frame,
- window,
- bookshelf,
- checkerboard,
- floor tiles,
- corridor,
- printed grid.

Try to include lines near the image borders, where distortion is usually stronger.


In [ ]:
CAMERA_ID = 0

cap = cv2.VideoCapture(CAMERA_ID)

if not cap.isOpened():
    raise RuntimeError(f"Could not open camera ID {CAMERA_ID}")

# Request the same resolution used during calibration.
if image_width is not None and image_height is not None:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, int(image_width))
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, int(image_height))

# Warm up camera.
for _ in range(10):
    ret, frame = cap.read()

ret, frame = cap.read()
cap.release()

if not ret:
    raise RuntimeError("Failed to capture image from webcam.")

print("Captured frame shape:", frame.shape)
show_image_bgr(frame, "Raw camera image")

# 7. Undistort the image

In [ ]:
undistorted_frame, new_camera_matrix, roi = undistort_image(
    frame,
    camera_matrix,
    dist_coeffs
)

print("New camera matrix:")
print(new_camera_matrix)
print("ROI:", roi)

show_image_bgr(undistorted_frame, "Undistorted image")

# 8. Compare raw and undistorted images side by side

In [ ]:
raw_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
undistorted_rgb = cv2.cvtColor(undistorted_frame, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.imshow(raw_rgb)
plt.title("Raw image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(undistorted_rgb)
plt.title("Undistorted image")
plt.axis("off")

plt.show()

# 9. Edge detection

We apply the same edge detector to both the raw and undistorted images.


In [ ]:
CANNY_LOW_THRESHOLD = 50
CANNY_HIGH_THRESHOLD = 150
BLUR_KERNEL_SIZE = 5

raw_gray = preprocess_for_edges(frame, blur_kernel_size=BLUR_KERNEL_SIZE)
undistorted_gray = preprocess_for_edges(undistorted_frame, blur_kernel_size=BLUR_KERNEL_SIZE)

raw_edges = detect_edges(
    raw_gray,
    low_threshold=CANNY_LOW_THRESHOLD,
    high_threshold=CANNY_HIGH_THRESHOLD
)

undistorted_edges = detect_edges(
    undistorted_gray,
    low_threshold=CANNY_LOW_THRESHOLD,
    high_threshold=CANNY_HIGH_THRESHOLD
)

show_gray_image(raw_edges, "Edges in raw image")
show_gray_image(undistorted_edges, "Edges in undistorted image")

# 10. Hough line detection

Important parameters:

| Parameter | Meaning |
|---|---|
| `threshold` | How many votes are needed to accept a line |
| `min_line_length` | Shorter line segments are ignored |
| `max_line_gap` | Small gaps between line pixels can be connected |

If too many lines are detected, increase `threshold` or `min_line_length`.

If too few lines are detected, decrease them.


In [ ]:
HOUGH_THRESHOLD = 80
MIN_LINE_LENGTH = 60
MAX_LINE_GAP = 10

raw_lines = detect_lines_hough(
    raw_edges,
    threshold=HOUGH_THRESHOLD,
    min_line_length=MIN_LINE_LENGTH,
    max_line_gap=MAX_LINE_GAP
)

undistorted_lines = detect_lines_hough(
    undistorted_edges,
    threshold=HOUGH_THRESHOLD,
    min_line_length=MIN_LINE_LENGTH,
    max_line_gap=MAX_LINE_GAP
)

summarize_lines(raw_lines, "Raw image lines")
summarize_lines(undistorted_lines, "Undistorted image lines")

raw_lines_image = draw_lines(frame, raw_lines)
undistorted_lines_image = draw_lines(undistorted_frame, undistorted_lines)

show_image_bgr(raw_lines_image, "Detected lines in raw image")
show_image_bgr(undistorted_lines_image, "Detected lines in undistorted image")

# 11. Compare line detection side by side

In [ ]:
raw_lines_rgb = cv2.cvtColor(raw_lines_image, cv2.COLOR_BGR2RGB)
undistorted_lines_rgb = cv2.cvtColor(undistorted_lines_image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.imshow(raw_lines_rgb)
plt.title(f"Raw image: {count_lines(raw_lines)} lines")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(undistorted_lines_rgb)
plt.title(f"Undistorted image: {count_lines(undistorted_lines)} lines")
plt.axis("off")

plt.show()

# 12. Optional: focus on long lines only

Short line segments can make the visualization noisy.

In robotics, long structural lines are often more useful.


In [ ]:
def filter_lines_by_length(lines, min_length_pixels=120):
    """Keep only lines longer than min_length_pixels."""
    if lines is None:
        return None

    filtered = []

    for line in lines:
        x1, y1, x2, y2 = line[0]
        length = np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

        if length >= min_length_pixels:
            filtered.append(line)

    if len(filtered) == 0:
        return None

    return np.array(filtered)


MIN_LONG_LINE_LENGTH = 120

raw_long_lines = filter_lines_by_length(raw_lines, MIN_LONG_LINE_LENGTH)
undistorted_long_lines = filter_lines_by_length(undistorted_lines, MIN_LONG_LINE_LENGTH)

summarize_lines(raw_long_lines, "Raw image long lines")
summarize_lines(undistorted_long_lines, "Undistorted image long lines")

raw_long_lines_image = draw_lines(frame, raw_long_lines, thickness=3)
undistorted_long_lines_image = draw_lines(undistorted_frame, undistorted_long_lines, thickness=3)

show_image_bgr(raw_long_lines_image, "Long lines in raw image")
show_image_bgr(undistorted_long_lines_image, "Long lines in undistorted image")

# 13. Optional: horizontal and vertical lines

Many indoor environments contain horizontal and vertical structures.

We can classify lines by their angle.


In [ ]:
def line_angle_degrees(line):
    """Return line angle in degrees."""
    x1, y1, x2, y2 = line[0]
    return np.degrees(np.arctan2(y2 - y1, x2 - x1))


def filter_lines_by_orientation(lines, mode="horizontal", tolerance_degrees=15):
    """
    Filter lines by orientation.

    mode can be:
    - "horizontal"
    - "vertical"
    """
    if lines is None:
        return None

    filtered = []

    for line in lines:
        angle = line_angle_degrees(line)

        # Normalize angle to [-90, 90].
        if angle > 90:
            angle -= 180
        if angle < -90:
            angle += 180

        if mode == "horizontal":
            if abs(angle) <= tolerance_degrees:
                filtered.append(line)
        elif mode == "vertical":
            if abs(abs(angle) - 90) <= tolerance_degrees:
                filtered.append(line)
        else:
            raise ValueError("mode must be 'horizontal' or 'vertical'")

    if len(filtered) == 0:
        return None

    return np.array(filtered)


horizontal_lines = filter_lines_by_orientation(undistorted_lines, mode="horizontal", tolerance_degrees=15)
vertical_lines = filter_lines_by_orientation(undistorted_lines, mode="vertical", tolerance_degrees=15)

horizontal_image = draw_lines(undistorted_frame, horizontal_lines, color=(255, 0, 0), thickness=3)
horizontal_vertical_image = draw_lines(horizontal_image, vertical_lines, color=(0, 255, 0), thickness=3)

print("Horizontal lines:", count_lines(horizontal_lines))
print("Vertical lines:", count_lines(vertical_lines))

show_image_bgr(horizontal_vertical_image, "Horizontal and vertical lines in undistorted image")

# 14. Live line detection demo

This cell opens the webcam and runs line detection live.

Press `q` to exit.

The live demo shows the result on the undistorted image.

> This cell opens an OpenCV GUI window. It may not work in all remote notebook environments.


In [ ]:
def run_live_line_detection(
    camera_id=0,
    camera_matrix=None,
    dist_coeffs=None,
    requested_width=None,
    requested_height=None,
    canny_low=50,
    canny_high=150,
    hough_threshold=80,
    min_line_length=60,
    max_line_gap=10
):
    if camera_matrix is None or dist_coeffs is None:
        raise ValueError("camera_matrix and dist_coeffs are required.")

    cap = cv2.VideoCapture(camera_id)

    if not cap.isOpened():
        raise RuntimeError(f"Could not open camera ID {camera_id}")

    if requested_width is not None and requested_height is not None:
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, int(requested_width))
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, int(requested_height))

    print("Live line detection started. Press 'q' in the image window to quit.")

    while True:
        ret, image = cap.read()

        if not ret:
            print("Failed to capture frame.")
            break

        undistorted, _, _ = undistort_image(image, camera_matrix, dist_coeffs)

        gray = preprocess_for_edges(undistorted, blur_kernel_size=5)
        edges = detect_edges(gray, canny_low, canny_high)
        lines = detect_lines_hough(
            edges,
            threshold=hough_threshold,
            min_line_length=min_line_length,
            max_line_gap=max_line_gap
        )

        output = draw_lines(undistorted, lines, thickness=2)

        text = f"Detected lines: {count_lines(lines)}"
        cv2.putText(
            output,
            text,
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.0,
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )

        cv2.imshow("Live calibrated line detection", output)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()


# Uncomment to run the live demo.

# run_live_line_detection(
#     camera_id=CAMERA_ID,
#     camera_matrix=camera_matrix,
#     dist_coeffs=dist_coeffs,
#     requested_width=image_width,
#     requested_height=image_height,
#     canny_low=CANNY_LOW_THRESHOLD,
#     canny_high=CANNY_HIGH_THRESHOLD,
#     hough_threshold=HOUGH_THRESHOLD,
#     min_line_length=MIN_LINE_LENGTH,
#     max_line_gap=MAX_LINE_GAP
# )

# 15. Experiment

## Experiment A: Raw vs undistorted lines

Use a scene with strong straight lines, such as a door frame or floor tiles.

Compare:

```text
Raw image line detection
vs.
Undistorted image line detection
```

Answer:

1. Do the detected lines look more geometrically consistent after undistortion?
2. Is the difference stronger near the image borders?
3. Are there fewer broken line segments after undistortion?

## Experiment B: Parameter tuning

Change these parameters:

```python
CANNY_LOW_THRESHOLD
CANNY_HIGH_THRESHOLD
HOUGH_THRESHOLD
MIN_LINE_LENGTH
MAX_LINE_GAP
```

Answer:

1. What happens if the Canny thresholds are too low?
2. What happens if the Hough threshold is too high?
3. What happens if `MIN_LINE_LENGTH` is too small?
4. Which settings work best for your scene?

## Experiment C: Robotics interpretation

Imagine a mobile robot using line detection for corridor following.

Answer:

1. Why could distorted lines be a problem?
2. Why does calibration help the robot understand the scene geometry?
3. Why is line detection alone not enough for full navigation?


# 16. Common problems

- Too many lines are detected

    Try:

    - increasing `HOUGH_THRESHOLD`,
    - increasing `MIN_LINE_LENGTH`,
    - increasing Canny thresholds,
    - using a simpler scene.

- Too few lines are detected

    Try:

    - decreasing `HOUGH_THRESHOLD`,
    - decreasing `MIN_LINE_LENGTH`,
    - decreasing Canny thresholds,
    - improving lighting.

- Image looks cropped after undistortion

    This is normal. Undistortion can create invalid border regions. The `alpha` parameter in `cv2.getOptimalNewCameraMatrix` controls the tradeoff between keeping the full image and removing invalid regions.

- Raw and undistorted images look almost the same

    Some laptop cameras have only mild distortion.
    Try placing straight lines near the image border.

- The line detector is unstable

    Line detection is sensitive to lighting, texture, and thresholds. This is normal. The goal is to understand the pipeline, not to create a perfect detector.


# References

- OpenCV camera calibration and undistortion: https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html
- OpenCV Canny edge detection: https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html
- OpenCV Hough line transform: https://docs.opencv.org/4.x/d9/db0/tutorial_hough_lines.html
